In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
folder_path = r"/storage/alplakes_test/geneva_100m_2025"
input_folder = os.path.join(folder_path, "outputs_swirl", "eddy_catalogues_final")

output_folder = os.path.join(folder_path, "outputs_swirl", "eddy_statistics")
os.makedirs(output_folder, exist_ok=True)

# Import lvl1 catalogue

In [ ]:
lvl1_csv_path = os.path.join(input_folder, "lvl1.csv")

In [ ]:
df_lvl1 = pd.read_csv(lvl1_csv_path)
df_lvl1 = df_lvl1.set_index('id', drop=False)
df_lvl1['date'] = pd.to_datetime(df_lvl1['date'])

In [ ]:
lake_mask = np.load(os.path.join(folder_path, "grid", "mask_lake.npy"))

In [ ]:
depths = pd.read_csv(os.path.join(folder_path, "grid", "depths.csv"))

# Number of eddies

In [ ]:
nb_eddy = df_lvl1.groupby(['date'])['id'].count()

In [ ]:
df_lvl1.groupby(['date'])['id'].count().iloc[7010]

In [ ]:
plt.figure(figsize=(8,5))
nb_eddy.plot()
plt.ylim(bottom=0)
plt.ylabel('Number of eddies [-]')
plt.xlabel("")
plt.grid(False)
plt.savefig(os.path.join(output_folder, "eddy_numbers.png"))

In [ ]:
def filter_by_depths(df, depth_min, depth_max):
    depth_filter = (
            (abs(df['depth_min_[m]']) >= abs(depth_max)) &
            (abs(df['depth_max_[m]']) <= abs(depth_min))
    )

    return df[depth_filter]

In [ ]:
depth_min = -10
depth_max = -0
df_lvl1_filtered_by_depth = filter_by_depths(df_lvl1, depth_min, depth_max)

In [ ]:
df_lvl1_filtered_by_depth[df_lvl1_filtered_by_depth['date']=='2025-05-12 03:30:00']

In [ ]:
nb_eddy_by_depth = df_lvl1_filtered_by_depth.groupby(['date'])['id'].count()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
nb_eddy_by_depth.plot()
plt.ylim(bottom=0)
plt.ylabel("Number of eddies")
plt.text(0.02, 0.98, f'{depth_min} to {depth_max}m', transform=plt.gca().transAxes, ha='left', va='top')
plt.xlabel("")
plt.savefig(os.path.join(output_folder, f"eddy_numbers_{depth_min}-{depth_max}m.png"))

In [ ]:
nb_eddy_by_depth.reset_index().to_csv(os.path.join(output_folder, f"eddy_numbers_{depth_min}-{depth_max}m.csv"))

# Surface statistics

In [ ]:
df_lvl1['surface_area_mean_[km2]'] = df_lvl1['surface_area_mean_[m2]'] / 1e6

In [ ]:
surface_timeserie = df_lvl1.groupby(['date'])['surface_area_mean_[km2]'].sum()

In [ ]:
plt.figure(figsize=(8,5))
surface_timeserie.plot()
plt.ylabel("Mean surface area [km$^2$]")
plt.xlabel('')
plt.savefig(os.path.join(output_folder, "surface_timeserie.png"))

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_lvl1['surface_area_mean_[km2]'], bins=50, kde=True)
plt.xlabel("Surface area [km2]")
plt.savefig(os.path.join(output_folder, "surface_statistics.png"))

# Volume statistics

In [ ]:
df_lvl1.head()

In [ ]:
df_lvl1['volume_[km3]'] = df_lvl1['volume_[m3]'] / 1e9

In [ ]:
volume_timeserie = df_lvl1.groupby(['date'])['volume_[km3]'].sum()

In [ ]:
volume_timeserie.to_csv(os.path.join(output_folder, 'eddy_volume_timeserie.csv'))

In [ ]:
plt.figure(figsize=(8,5))
volume_timeserie.plot()
plt.ylabel("Volume [km$^3$]")
plt.xlabel('')
plt.grid(False)
plt.savefig(os.path.join(output_folder, "volume_timeserie.png"))

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_lvl1['volume_[km3]'], bins=50)
plt.xlabel("Volume [km$^3$]")
plt.savefig(os.path.join(output_folder, "volume_statistics.png"))

In [ ]:
rho_w = 1000 # kg/m3
df_lvl1['ke_density_[J/kg]'] = 1e6 * df_lvl1['kinetic_energy_eddy_[MJ]']/(df_lvl1['volume_[m3]']*rho_w)

In [ ]:
df_lvl1['ke_density_[J/kg]'].to_csv(os.path.join(folder_path, "outputs_swirl", "ke_eddy", "eddy_ke_in_Jperkg.csv"))

In [ ]:
np.log10(df_lvl1['ke_density_[J/kg]'][df_lvl1['ke_density_[J/kg]']>0])

In [ ]:
# Violin plot
plt.figure(figsize=(6, 8))
sns.violinplot(data=np.log10(df_lvl1['ke_density_[J/kg]'][df_lvl1['ke_density_[J/kg]']>0]),
               inner='quartile',
               color='tab:blue')  # inner='quartile' shows median & quartiles
plt.title('Violin Plot of Eddy Kinetic Energy density [J/kg] (log10)')
plt.ylabel('log10(Kinetic Energy density) [J/kg]')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# Height statistics

In [ ]:
depths = pd.read_csv(os.path.join(folder_path, "grid", "depths.csv"))

In [ ]:
depths.head()

In [ ]:
df_lvl1['height_[m]'] = df_lvl1['depth_max_[m]'] - df_lvl1['depth_min_[m]']

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_lvl1['height_[m]'], bins=50, kde=True)
plt.xlabel("Height [m]")
#ax.xaxis.set_major_locator(MultipleLocator(1))
plt.savefig(os.path.join(output_folder, "height_statistics.png"))

# Depth statistics

In [ ]:
df_lvl1['depth_max_[m]']

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_lvl1['depth_max_[m]'], bins=50, kde=True)
plt.xlabel("Depth max [m]")
#ax.xaxis.set_major_locator(MultipleLocator(1))
plt.savefig(os.path.join(output_folder, "depth_max.png"))

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_lvl1['depth_min_[m]'], bins=50, kde=True)
plt.xlabel("Depth min [m]")
#ax.xaxis.set_major_locator(MultipleLocator(1))
plt.savefig(os.path.join(output_folder, "depth_min.png"))

In [ ]:
df_lvl1['time_index'].max()

In [ ]:
import numpy as np

# Define depth grid
depth_bins = np.arange(0, -60, -1)  # adjust max depth + resolution

occupancy = np.zeros([int(df_lvl1['time_index'].max()+1), len(depth_bins)-1])

for _, row in df_lvl1.iterrows():
    mask = (depth_bins[:-1] >= row["depth_min_[m]"]) & (depth_bins[:-1] <= row["depth_max_[m]"])
    occupancy[int(row['time_index']), mask] += 1

In [ ]:
plt.plot(occupancy, depth_bins[:-1])
plt.xlabel("Fraction of eddies")
plt.ylabel("Depth")
plt.title("Eddy Occurrence in Water Column")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.imshow(
    occupancy.T,
    aspect='auto',
    origin='lower',
    extent=[
        0, occupancy.shape[0],          # time axis
        depth_bins[0], depth_bins[-1]   # depth axis
    ]
)

plt.colorbar(label='Eddy Occupancy')
plt.gca().invert_yaxis()
plt.xlabel('Time index')
plt.ylabel('Depth (m)')
plt.title('Depth Occupancy Over Time')

plt.show()